# Ставки и M2

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.stats.api import durbin_watson, acorr_breusch_godfrey # DW & LM-тесты
from statsmodels.iolib.summary2 import summary_col # вывод подгонки

import matplotlib.pyplot as plt

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

## Данные

Из БД [FRED](https://fred.stlouisfed.org/) загрузим месячные данные

* M2 (M2SL)
* 3-Month Treasury Bill Secondary Market Rate (TB3MS)
* Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity (GS10)
* Market Yield on U.S. Treasury Securities at 1-Year Constant Maturity (GS1)
* Moody's Seasoned Aaa Corporate Bond Yield (AAA)
* Moody's Seasoned Baa Corporate Bond Yield (BAA)

с 2000-01 по 2023-12

In [ ]:
# ряды FRED и период
series = ['TB3MS', 'GS1', 'GS10', 'AAA', 'BAA', 'M2SL']
start_date = '2000-01-01'
end_date = '2023-12-31'
# каждый ряд читаем из CSV-файла FRED средствами pandas
url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id='
data = pd.concat([pd.read_csv(url+s, index_col=0, parse_dates=True) for s in series], axis=1)
data = data.loc[start_date:end_date]
data

In [ ]:
# график исходных данных
data.plot(subplots=True)
plt.show()

In [ ]:
# график дифференцированных данных
data.diff().plot(subplots=True)
plt.show()

In [ ]:
# спреды ставок и логарифм M2
spreads = pd.DataFrame({'SpreadTres': data['GS10']-data['TB3MS'],
                        'SpreadMoodys': data['BAA']-data['AAA'],
                        'logM2SL': np.log(data['M2SL'])})
# график спредов и логарифма M2
spreads.plot(subplots=True)
plt.show()

In [ ]:
# график дифференцированных спредов и логарифма M2
spreads.diff().plot(subplots=True)
plt.show()

## Модель

In [ ]:
# первые разности
spreads.diff()

In [ ]:
# набор данных для модели: первые разности без пропусков
df = spreads.diff().dropna()

In [ ]:
# спецификация модели через формулу
mod = ols(formula='SpreadMoodys~SpreadTres+logM2SL', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res = mod.fit()

### Диагностика

In [ ]:
# LM-тест на автокорреляцию 3-го порядка
lm, lmpval, fval, fpval = acorr_breusch_godfrey(res, nlags=3)
lm, lmpval

In [ ]:
# DW-статистика
durbin_watson(res.resid)

### Подгонка

In [ ]:
# спецификация модели через формулу
mod = ols(formula='SpreadMoodys~SpreadTres+logM2SL', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res_ols = mod.fit()
# число лагов для HAC-оценки
L = int(4*(mod.nobs/100)**(2/9))
# подгонка модели с робастной HAC-оценкой ковариационной матрицы
res_hac = mod.fit(cov_type='HAC', cov_kwds={'maxlags': L}, use_t=True)

In [ ]:
# неробастные и робастные (HAC) стандартные ошибки
summary_col(results=[res_ols, res_hac], model_names=['OLS-CME', 'HAC-CME'], stars=True)

In [ ]:
# значимость регрессии: P-значения неробастного и робастного F-теста
res_ols.f_pvalue, res_hac.f_pvalue